# Fig. 3: Asymptotic coefficients

*Code cells: MIT licence (`LICENSE`). Explanatory text and the figure: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang).*

In [1]:
import sys, json, math
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import plot_main_figures as pmf
pd.set_option('display.precision', 8)

## What the figure shows

Four tests of the asymptotic coefficients derived in the article.

* **(a)** The short-chain deficit $(\lambda/2-F)/\lambda^2$ from refined spectral solutions
  (angular orders up to 512 with first-alias-corrected inversion, `data/stiff_alias_summary.csv`)
  approaches $37/768$ as $\lambda\to0$.
* **(b)** The excess of the span over the mean projected end-to-end distance,
  $[X-\langle|\mathbf R\cdot\hat{\mathbf e}|\rangle]/(p\lambda^2)$, from polygonal path sampling
  with von Mises–Fisher tangent steps (`data/endpoint_excursion_diagnostic.csv`), compared with the
  predicted $9/256$. Bars are two sampling standard errors.
* **(c)** The deficit of the spectral span relative to the leading Gaussian law,
  $4\sqrt{\lambda/3\pi}-F$, plotted against $\lambda^{-1/2}$ for $\lambda\ge5$; the intercept is
  $2z_M$ and the slope is $-B$.
* **(d)** The wall offset of the depletion layer, $2R_g/\sqrt\pi-\delta$ with $\delta=X/2$, from the same
  spectral values; it approaches $z_M$ with the slope $-(B/2+\sqrt{3/\pi})$ (the wall-offset expansion of the article).

In [2]:
stiff = pd.read_csv(ROOT / 'data' / 'stiff_alias_summary.csv')
stiff['deficit/lambda^2'] = (stiff['lambda']/2 - stiff.F_extrap) / stiff['lambda']**2
print('37/768 =', 37/768)
stiff[['lambda', 'F_extrap', 'extrapolation_sensitivity', 'deficit/lambda^2']]

37/768 = 0.048177083333333336


,lambda,F_extrap,extrapolation_sensitivity,deficit/lambda^2
0,0.0125,0.00624248,9.94280786e-12,0.04810784
1,0.0250,0.01246998,1.93635212e-12,0.04803877
2,0.0500,0.02488025,1.08249867e-12,0.04790109
3,0.0750,0.03723133,2.76637047e-12,0.04776402
4,0.1000,0.04952372,4.20451868e-12,0.04762756
5,0.2000,0.09811649,8.39424363e-12,0.04708779


In [3]:
ex = pd.read_csv(ROOT / 'data' / 'endpoint_excursion_diagnostic.csv')
ex = ex[ex['lambda'] <= 0.1].sort_values(['lambda', 'steps']).groupby('lambda').tail(1)
print('9/256 =', 9/256)
ex[['lambda', 'steps', 'n', 'excursion_over_lambda2', 'excursion_se']]

9/256 = 0.03515625


,lambda,steps,n,excursion_over_lambda2,excursion_se
0,0.005,512,160000,0.03494475,0.00037573
1,0.010,512,160000,0.03519042,0.00031264
2,0.020,512,80000,0.03487143,0.00036764
3,0.050,512,80000,0.03492859,0.00028682
5,0.100,2048,60000,0.03470485,0.00026917


In [4]:
ref = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
big = ref[ref['lambda'] >= 5]
u = 1/np.sqrt(big['lambda']); off = pmf.A_GAUSS*np.sqrt(big['lambda']) - big.F_continuum
print('2 z_M =', 2*pmf.Z_M, '   B =', pmf.B_COEF)
# the line in panel (c) is NOT a fit to these points; it uses z_M (transport theory) and B (small-s analysis)
print('max |offset - (2z_M - B/sqrt(lambda))| for lambda>=5 :', np.max(np.abs(off - (2*pmf.Z_M - pmf.B_COEF*u))))

2 z_M = 1.43642328    B = 0.6468
max |offset - (2z_M - B/sqrt(lambda))| for lambda>=5 : 0.0052259601964805835


In [5]:
g = pmf.rg2_over_p2(big['lambda'].to_numpy())
wall = 2*np.sqrt(g)/np.sqrt(np.pi) - big.F_continuum.to_numpy()/2
pd.DataFrame({'lambda': big['lambda'], 'wall offset / p': wall,
              'z_M - (B/2 + sqrt(3/pi))/sqrt(lambda)': pmf.Z_M - (pmf.B_COEF/2 + np.sqrt(3/np.pi))*u}).iloc[::6]

,lambda,wall offset / p,z_M - (B/2 + sqrt(3/pi))/sqrt(lambda)
29,5.0,0.24060202,0.13656339
35,12.0,0.37240444,0.34275931
41,50.0,0.53781545,0.53427831
47,200.0,0.62668799,0.62624498
53,1000.0,0.67712252,0.67708290
59,7500.0,0.70319563,0.70319355
65,75000.0,0.71345547,0.71346250


## Render the figure

In [6]:
fig = pmf.figure3()
fig

<Figure size 677.165x448.819 with 4 Axes>